# 04 Backends and SDKs

<a href="https://colab.research.google.com/github/bsc-wdc/qdislib/blob/main/notebooks/04_backends.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a> <a href="https://mybinder.org/v2/gh/bsc-wdc/qdislib/main?labpath=notebooks/04_backends.ipynb"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"></a>

Two separate choices: the **SDK a circuit is written in** (Qiskit, Qibo, CUDA-Q, PennyLane, QiliSDK), and the
**backend** where its subcircuits run (`backend=`). They need not match. This notebook runs Qiskit and Qibo; the code for the others is shown
as text because they are not in this notebook's install.

In [1]:
# Installs Qdislib on Colab or Binder; does nothing where Qdislib 2 is installed.
try:
    import Qdislib
    assert Qdislib.__version__.startswith("2.")
except (ImportError, AssertionError):
    %pip install -q "qdislib[qibo,qiskit]>=2.0"

In [2]:
import logging

logging.getLogger("qibo.config").setLevel(logging.WARNING)  # Qibo logs its backend on every set-up

from qiskit import QuantumCircuit
import Qdislib as qd


def example_circuit():
    """Five qubits in a line, joined by four CZ gates."""
    qc = QuantumCircuit(5)
    for qubit, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3]):
        qc.ry(angle, qubit)
    for qubit in range(4):
        qc.cz(qubit, qubit + 1)
    for qubit, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9]):
        qc.rx(angle, qubit)
    return qc


circuit = example_circuit()
cut = ["CZ_7"]

## What `backend=` takes

- `None` (the default): the circuit's own SDK's simulator;
- an SDK name: `"qiskit"` (Aer), `"qibo"`, `"cudaq"`, `"pennylane"`, `"qilisdk"`;
- a registered backend name: `qd.available_backends()` lists them;
- a backend object from `qd.build_backend(name, qd.BackendConfig(...))`, or your own (notebook 08).

The registry lists backends without importing their SDKs, so an SDK that is not installed only matters when you use it.

In [3]:
print(qd.available_backends())

print("default (the circuit's SDK):", qd.gate_cutting(circuit, cut, shots=8192, seed=1))
print('backend="qiskit":           ', qd.gate_cutting(circuit, cut, shots=8192, seed=1, backend="qiskit"))
print('backend="qibo":             ', qd.gate_cutting(circuit, cut, shots=8192, seed=1, backend="qibo"))
print("exact:                      ", qd.analytical_solution(circuit, "ZZZZZ"))

['AER_BACKEND', 'QIBO_BACKEND', 'CUDAQ_BACKEND', 'PENNYLANE_BACKEND', 'QILISDK_BACKEND', 'IQM_BACKEND', 'BSC_QPU_BACKEND', 'IBM_BACKEND']
default (the circuit's SDK): 0.23241020739078522


backend="qiskit":            0.23241020739078522


backend="qibo":              0.24546118080615997
exact:                       0.23901960083420604


## `shots=None` and `seed`

`shots=None` is exact: each fragment's statevector, on Qdislib's own small simulator. It is the fastest way to check a cut, and nothing is random.
With a registered backend or backend object, `shots=None` is still a Qiskit statevector, not a run on that backend, and Qdislib warns once to say so.

`seed` makes a simulator's shots reproducible; each subcircuit gets its own stream.

In [4]:
print("seed=3 twice:", qd.gate_cutting(circuit, cut, shots=1024, seed=3), qd.gate_cutting(circuit, cut, shots=1024, seed=3))
print("no seed:     ", qd.gate_cutting(circuit, cut, shots=1024), qd.gate_cutting(circuit, cut, shots=1024))
print("exact:       ", qd.gate_cutting(circuit, cut, shots=None))

seed=3 twice: 0.2117900848388672 0.2117900848388672
no seed:      0.24193096160888672 0.2135334014892578
exact:        0.23901960083420587


## The same circuit in another SDK

Inside Qdislib every circuit becomes the same DAG, whose gates are named as in OpenQASM 3 (`cx`, `cz`, `rzz`, `u`, ...).
So a cut names a gate the same way in every SDK, by its position counting from 1, and `CX_7`, `CNOT_7` and `cx_7` are the same gate. Naming a gate that is not there says what is there.

In [5]:
from qibo import gates, models

in_qibo = models.Circuit(5)
for qubit, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3]):
    in_qibo.add(gates.RY(qubit, angle))
for qubit in range(4):
    in_qibo.add(gates.CZ(qubit, qubit + 1))
for qubit, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9]):
    in_qibo.add(gates.RX(qubit, angle))

print("Qibo circuit, cut found:    ", qd.find_cut(in_qibo, max_qubits=3, wire_cut=False))
print("Qibo circuit on Aer:        ", qd.gate_cutting(in_qibo, cut, shots=8192, seed=1, backend="qiskit"))
print("Qibo circuit, exact:        ", qd.gate_cutting(in_qibo, cut, shots=None))

try:
    qd.gate_cutting(circuit, ["CX_7"])
except Exception as error:
    print(type(error).__name__, str(error)[:120])

Qibo circuit, cut found:     ['CZ_7']
Qibo circuit on Aer:         0.23241020739078522
Qibo circuit, exact:         0.23901960083420587
ValueError No gate 'CX_7': gate 7 of this circuit is 'CZ_7' (cz).


## Writing the subcircuits in any SDK

A subcircuit list is SDK-neutral. `subs.circuits(software)` builds its circuits in the SDK you ask for.

In [6]:
subs = qd.gate_cutting_subcircuits(circuit, cut)
for software in ("qiskit", "qibo"):
    print(software, type(subs.circuits(software)[0]))

qiskit <class 'qiskit.circuit.quantumcircuit.QuantumCircuit'>
qibo <class 'qibo.models.circuit.Circuit'>


## Circuits from elsewhere: `from_qasm`

Anything that exports OpenQASM 2 can come in through `from_qasm`, which builds a circuit in the SDK you name.

In [7]:
from qiskit.qasm2 import dumps

text = dumps(circuit)
print(text[:120], "...")
imported = qd.from_qasm(text, software="qibo")
print(type(imported), qd.analytical_solution(imported, "ZZZZZ"))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[5];
ry(0.8) q[0];
ry(0.5) q[1];
ry(0.6) q[2];
ry(0.4) q[3];
ry(0.3) q[4];
cz  ...
<class 'qibo.models.circuit.Circuit'> 0.23901960083420604


## Dense unitary gates

A Qiskit `UnitaryGate` can stay in a cut circuit as it is. It runs on Qiskit/Aer, Qibo and Qdislib's exact simulator; CUDA-Q and PennyLane
refuse it with an error that says so. This matters for a circuit built from multi-qubit isometries, such as an MPS-loaded state: its dense gates are simulated
directly, instead of being decomposed into a very large number of CNOTs first.

The cut itself must be on a gate that crosses between fragments. A dense gate can be cut as any other two-qubit gate.

In [8]:
from qiskit.circuit.library import UnitaryGate
from scipy.stats import unitary_group

dense = QuantumCircuit(4)
dense.append(UnitaryGate(unitary_group.rvs(4, random_state=1)), [0, 1])
dense.cz(1, 2)
dense.append(UnitaryGate(unitary_group.rvs(4, random_state=2)), [2, 3])

print(dense.count_ops())
print("exact:", qd.analytical_solution(dense, "ZZZZ"))
print("cut, exact pieces:", qd.gate_cutting(dense, ["CZ_2"], "ZZZZ", shots=None))
print("cut, Aer:         ", qd.gate_cutting(dense, ["CZ_2"], "ZZZZ", shots=100_000, seed=1, backend="qiskit"))
print("cut, Qibo:        ", qd.gate_cutting(dense, ["CZ_2"], "ZZZZ", shots=100_000, seed=1, backend="qibo"))

OrderedDict({'unitary': 2, 'cz': 1})
exact: 0.002117304701836409
cut, exact pieces: 0.002117304701836506


cut, Aer:          0.0023644418000000195


cut, Qibo:         0.0034989379999999792


## GPUs, and the other SDKs

These cells are **not run here**: they need a GPU or an SDK this notebook does not install. The code below was run against this version of Qdislib on a machine that has the three SDKs, except the two GPU lines.

```python
# CUDA-Q (pip install "qdislib[cudaq]")
import cudaq

@cudaq.kernel
def kernel():
    q = cudaq.qvector(4)
    ry(0.8, q[0])
    z.ctrl(q[0], q[1])
    ry(0.5, q[1])
    z.ctrl(q[1], q[2])
    ry(0.6, q[2])
    z.ctrl(q[2], q[3])

cut = qd.find_cut(kernel, max_qubits=2, wire_cut=False)
value = qd.gate_cutting(kernel, cut, backend="cudaq", shots=8192)
```

```python
# PennyLane (pip install "qdislib[pennylane]")
import pennylane as qml

with qml.tape.QuantumTape() as tape:
    qml.RY(0.8, wires=0)
    qml.CZ(wires=[0, 1])
    qml.RY(0.5, wires=1)
    qml.CZ(wires=[1, 2])

value = qd.gate_cutting(tape, ["CZ_4"], backend="pennylane", shots=8192)
```

```python
# QiliSDK (pip install "qdislib[qilisdk]")
from qilisdk.digital import CZ, RY, Circuit

c = Circuit(3)
c.add(RY(0, theta=0.8))
c.add(CZ(0, 1))
c.add(RY(1, theta=0.5))
c.add(CZ(1, 2))

value = qd.gate_cutting(c, ["CZ_4"], backend="qilisdk", shots=8192)
```

```python
# A GPU: Aer's cuStateVec (needs qiskit-aer-gpu), and CUDA-Q's nvidia target
gpu = qd.build_backend("AER_BACKEND", qd.BackendConfig(extra={"use_gpu": True}))
gpu = qd.build_backend("CUDAQ_BACKEND", qd.BackendConfig(extra={"target": "nvidia"}))
value = qd.gate_cutting(circuit, cut, backend=gpu)
```

Under PyCOMPSs a backend on a GPU runs its circuits in tasks that ask for one GPU: see notebook 09.

Next: **05 Observables**.